In [1]:
# ==========================================
# 1. Import Libraries
# ==========================================

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


# ==========================================
# 2. Load Titanic Dataset
# ==========================================

df = sns.load_dataset("titanic")


# ==========================================
# 3. Select Features and Target
# ==========================================

X = df[['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare']]
y = df['survived']


# ==========================================
# 4. Convert Categorical Variable
# ==========================================

X = pd.get_dummies(X, columns=['sex'])


# ==========================================
# 5. Handle Missing Values
# ==========================================

X['age'] = X['age'].fillna(X['age'].mean())


# ==========================================
# 6. Train-Test Split
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ==========================================
# 7. Define Models
# ==========================================

models = {

    'Logistic Regression': LogisticRegression(
        max_iter=1000
    ),

    'SVM': SVC(),

    'Decision Tree': DecisionTreeClassifier(
        random_state=42
    ),

    'Random Forest': RandomForestClassifier(
        random_state=42
    ),

    'KNN': KNeighborsClassifier()
}


# ==========================================
# 8. Define Parameter Grids
# ==========================================

param_grids = {

    'Logistic Regression': {
        'C': [0.01, 0.1, 1, 10, 100],
        'solver': ['liblinear', 'lbfgs']
    },

    'SVM': {
        'C': [0.1, 1, 10, 100],
        'kernel': ['linear', 'rbf'],
        'gamma': ['scale', 'auto']
    },

    'Decision Tree': {
        'criterion': ['gini', 'entropy'],
        'max_depth': [3, 5, 7, 10, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },

    'Random Forest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [5, 10, 15, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2']
    },

    'KNN': {
        'n_neighbors': [3, 5, 7, 9, 11],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan']
    }
}


# ==========================================
# 9. Grid Search CV
# ==========================================

best_models = {}
results = []

for model_name, model in models.items():

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grids[model_name],
        cv=5,
        scoring='accuracy',
        n_jobs=-1
    )

    # Train GridSearchCV
    grid_search.fit(X_train, y_train)

    # Best model
    best_model = grid_search.best_estimator_

    # Save best model
    best_models[model_name] = best_model

    # Predictions using best model
    y_pred = best_model.predict(X_test)

    # Evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    # Store results
    results.append([
        model_name,
        grid_search.best_score_,
        accuracy,
        precision,
        recall,
        f1
    ])

    # Print best parameters
    print("\nBest Parameters:")
    print(grid_search.best_params_)

    # Print CV score
    print("\nBest Cross-Validation Accuracy:")
    print(f"{grid_search.best_score_:.3f}")

    # Print best model
    print("\nBest Model:")
    print(best_model)

    # Print test scores
    print("\nTest Set Performance:")
    print(f"Accuracy  : {accuracy:.3f}")
    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1 Score  : {f1:.3f}")


# ==========================================
# 10. Create Results DataFrame
# ==========================================

results_df = pd.DataFrame(
    results,
    columns=[
        'Model',
        'CV Accuracy',
        'Test Accuracy',
        'Precision',
        'Recall',
        'F1 Score'
    ]
)


# ==========================================
# 11. Sort Models by Test Accuracy
# ==========================================

results_df = results_df.sort_values(
    by='Test Accuracy',
    ascending=False
)


# ==========================================
# 12. Display Final Comparison
# ==========================================

print("\n")
print("=" * 80)
print("FINAL MODEL COMPARISON")
print("=" * 80)

print(
    results_df.to_string(
        index=False,
        formatters={
            'CV Accuracy': '{:.3f}'.format,
            'Test Accuracy': '{:.3f}'.format,
            'Precision': '{:.3f}'.format,
            'Recall': '{:.3f}'.format,
            'F1 Score': '{:.3f}'.format
        }
    )
)


# ==========================================
# 13. Best Overall Model
# ==========================================

best_model_name = results_df.iloc[0]['Model']
best_model = best_models[best_model_name]

print("\n")
print("=" * 80)
print("BEST OVERALL MODEL")
print("=" * 80)

print(f"Model: {best_model_name}")

print("\nBest Parameters:")
print(best_model.get_params())

print("\nBest Model:")
print(best_model)


Logistic Regression

Best Parameters:
{'C': 1, 'solver': 'liblinear'}

Best Cross-Validation Accuracy:
0.796

Best Model:
LogisticRegression(C=1, max_iter=1000, solver='liblinear')

Test Set Performance:
Accuracy  : 0.804
Precision : 0.793
Recall    : 0.667
F1 Score  : 0.724

SVM

Best Parameters:
{'C': 100, 'gamma': 'scale', 'kernel': 'rbf'}

Best Cross-Validation Accuracy:
0.794

Best Model:
SVC(C=100)

Test Set Performance:
Accuracy  : 0.782
Precision : 0.727
Recall    : 0.696
F1 Score  : 0.711

Decision Tree

Best Parameters:
{'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 10}

Best Cross-Validation Accuracy:
0.827

Best Model:
DecisionTreeClassifier(max_depth=5, min_samples_split=10, random_state=42)

Test Set Performance:
Accuracy  : 0.760
Precision : 0.760
Recall    : 0.551
F1 Score  : 0.639

Random Forest

Best Parameters:
{'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 200}

Best Cross